In [1]:
import numpy as np
from Autograd import Tensor
from Model import Linear,Sequential,Embedding
from Loss import cross_entropy_loss
from Activation import ReLU
from Training import training
from Optimiser import SGD,Adam
import matplotlib.pyplot as plt

Machine learning models only work with numbers, not words. A common approach is to assign different words a unique ID randomly; from here we try to associate each ID with some vector in d dimensional space, where d can be chosen.

A naive approach could be to use hot-one vectors, but this has a major issue. If for example the word 'cat' was associated with the vector $e_1$, 'dog' $e_2$ and 'pizza' $e_3$ ($e_i$ is a standard basis vector). There is no notion that dog and cat are more closely linked than cat and pizza (same distance etc.), despite 2 of them being living animals and the other being an italian staple food.

A popular technique to overcome this is to use an embedding. The ID assigned to each word is used to find an associated vector in a lookup table. In the context of an MLP to predict the next word, we can think of the softmaxed output as a conditional probability distribution over the possible words given the context (previous words). The parameters can hence be learned by gradient descent using the same loss function as the other MLP parameters.

This approach can be applied to many contexts, not just words. For example, given a set of words, we could train an MLP to predict the next letter, where the letters of the alphabet are indexed 0 to 25.

In [8]:
X=Tensor(np.array([0,1,2,0,1]))
Y=Tensor(np.array([1,2,0,1,3]))
model=Sequential([Embedding(4,8),Linear(8,16),ReLU(),Linear(16,4)])
loss_fn=cross_entropy_loss
optimiser=Adam(model.parameters())
loss=training(X,model,Y,loss_fn,1000,optimiser,5)
loss[-1]


np.float64(0.27787636729385196)

In [10]:
words = ["i","like","cats","dogs"]
vocab = {"i":0,"like":1,"cats":2,"dogs":3}
idx_to_word = {i:word for word,i in vocab.items()}
for i,word in enumerate(words):
    x=Tensor(np.array([i]))
    logits=model.forward_prop(x)
    probs=logits.softmax(axis=1)
    # get index with highest probability
    predicted_idx=np.argmax(probs.data, axis=1)[0]
    # convert index back to word
    predicted_word=idx_to_word[predicted_idx]
    print(f"Input: {word}")
    print(f"Probabilities: {probs.data[0]}")
    print(f"Predicted next word: {predicted_word}")
    print()

Input: i
Probabilities: [6.39093447e-05 9.99094595e-01 7.03400302e-04 1.38094941e-04]
Predicted next word: like

Input: like
Probabilities: [2.33564006e-04 3.90848232e-04 4.99502606e-01 4.99872982e-01]
Predicted next word: dogs

Input: cats
Probabilities: [9.99979198e-01 7.54910287e-06 1.05139455e-05 2.73862957e-06]
Predicted next word: i

Input: dogs
Probabilities: [0.02463387 0.07207463 0.41349441 0.48979709]
Predicted next word: dogs

